# CCM-109 - Tópicos especiais de IA - Deep Learning

## Pipeline de detecção de deepfake

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jtlimo/ccm-109/blob/main/prediction.ipynb)

---

### Índice

1. [Configuração](#1-configuração)
2. [Dataset FF++](#2-dataset-faceforensics)
3. [Funções auxiliares](#3-funções-auxiliares)
4. [Treino](#4-treino)

## 1.Configuração

In [ ]:
%pip install -q tensorflow scikit-learn keras-hub


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# @title Configurações

import os
import glob
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path
import keras_hub


model_files = glob.glob("model.*-*.keras")
if not model_files:
    print("⚠️ Nenhum modelo encontrado! Verifique se o treinamento foi executado.")
    model_files = ["model.00-0.0000.keras"]
model_files.sort(
    key=lambda x: float(x.split('-')[1].replace('.keras', '')), 
    reverse=True
)

model_widget = widgets.Dropdown(
    options=model_files,
    value=model_files[0],
    description='Modelo:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

dataset_widget = widgets.Dropdown(
    options=['Celeb-DF', 'DeeperForensics', 'Custom', 'FF++'],
    value='FF++',
    description='Dataset:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

backbone_widget = widgets.Dropdown(
    options=['Original (congelado)', 'Fine-tuned'],
    value='Original (congelado)',
    description='Backbone:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

threshold_widget = widgets.FloatSlider(
    value=0.5,
    min=0.1,
    max=0.9,
    step=0.05,
    description='Threshold:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

batch_size_widget = widgets.IntSlider(
    value=32,
    min=8,
    max=128,
    step=8,
    description='Batch size:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

aggregation_widget = widgets.Dropdown(
    options=['median', 'mean', 'max', 'trimmed_mean'],
    value='median',
    description='Agregação:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

trim_percent_widget = widgets.IntSlider(
    value=10,
    min=0,
    max=50,
    step=5,
    description='Trim %:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='400px')
)

print("=" * 60)
print("CONFIGURAÇÕES DE GENERALIZAÇÃO")
print("=" * 60)
display(
    model_widget, 
    dataset_widget, 
    backbone_widget,
    threshold_widget, 
    batch_size_widget,
    aggregation_widget,
    trim_percent_widget
)

def get_config():    
    cfg = {
        "model_path": model_widget.value,
        "dataset": dataset_widget.value,
        "use_finetuned": backbone_widget.value == 'Fine-tuned',
        "threshold": threshold_widget.value,
        "batch_size": batch_size_widget.value,
        "aggregation": aggregation_widget.value,
        "trim_percent": trim_percent_widget.value,
        
        "img_size": pi.SIGLIP2_INPUT_SIZE,  
        "confidence": pi.CONFIDENCE_THRESHOLD,
        "min_face_size": pi.MIN_FACE_SIZE,
        "skip_frames": pi.SKIP_FRAMES,
        "max_frames": pi.MAX_FRAMES_PER_VIDEO,
        "max_read_limit": pi.MAX_READ_LIMIT,
        "jpg_quality": pi.JPG_QUALITY,
    }
    
    DATASET_PATHS = {
        'Celeb-DF': '/dataset/Celeb-DF',
        'DeeperForensics': '/dataset/DeeperForensics',
        'Custom': '/dataset/Custom',
        'FF++': '/dataset/FF',
    }
    
    base_path = DATASET_PATHS.get(cfg['dataset'], '/dataset/FF')
    cfg["pred_output_dir"] = os.path.join(base_path, "predictions")
    cfg["npy_dir"] = os.path.join(base_path, "npy")
    
    return cfg

CFG = get_config()
print("\n" + "=" * 60)
print("CFG ATUAL:")
print("=" * 60)
for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

In [ ]:
# @title Imports

from pathlib import Path
import numpy as np
import cv2

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Desativa GPU

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt

import preprocess_img as pi
import grad_cam as gc

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

2026-08-05 21:20:24.306325: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785975627.722102 1091963 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2423 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1050, pci bus id: 0000:01:00.0, compute capability: 6.1
2026-08-05 21:20:27.952309: W external/local_xla/xla/service/gpu/llvm_gpu_backend/default/nvptx_libdevice_path.cc:41] Can't find libdevice directory ${CUDA_DIR}/nvvm/libdevice. This may result in compilation or runtime failures, if the program we try to run uses routines from libdevice.
Searched for CUDA in the following directories:
  ./cuda_sdk_lib
  ipykernel_launcher.runfiles/cuda_nvcc
  ipykernel_launcher.runfiles/cuda_nvdisasm
  ipykernel_laun

TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Funções auxiliares

In [ ]:
def load_original_frame(jpg_path, target_size=224):
    img = cv2.imread(str(jpg_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
    return img.astype(np.float32) / 255.0


def load_original_frames(frame_paths, target_size=224):
    frames = [load_original_frame(p, target_size) for p in frame_paths]
    return np.stack(frames, axis=0)

In [ ]:
def aggregate_mean(preds):
    return float(np.mean(preds))


def aggregate_median(preds):
    return float(np.median(preds))


def aggregate_vote(preds, threshold=0.5):
    return float(np.mean(preds > threshold))


def aggregate_trimmed_mean(preds, trim_percent=10):
    lower = np.percentile(preds, trim_percent)
    upper = np.percentile(preds, 100 - trim_percent)
    trimmed = preds[(preds >= lower) & (preds <= upper)]
    return float(np.mean(trimmed))


def aggregate_video(preds, method="median", threshold=0.5, trim_percent=10):
    if method == "mean":
        return aggregate_mean(preds)
    elif method == "median":
        return aggregate_median(preds)
    elif method == "vote":
        return aggregate_vote(preds, threshold)
    elif method == "trimmed_mean":
        return aggregate_trimmed_mean(preds, trim_percent)
    else:
        raise ValueError(f"Método desconhecido: {method}")


def classify_video(video_score, threshold=0.5):
    return "fake" if video_score > threshold else "real"

# 3. Pré-processamento

In [ ]:
def load_preprocessed_frames(video_name, frames_dir):
    video_dir = Path(frames_dir) / video_name
    npy_files = sorted(video_dir.glob("*_siglip2.npy"))

    if not npy_files:
        return None

    frames = [np.load(f) for f in npy_files]
    return np.stack(frames, axis=0)


def preprocess_video_for_prediction(video_path, output_dir=None):
    output_dir = output_dir or CFG["pred_output_dir"]
    video_path = Path(video_path)
    video_name = video_path.stem

    n_faces = pi.process_video(video_path, output_dir)

    if n_faces == 0:
        print(f"[AVISO] Nenhuma face detectada em {video_path.name}")
        return {"tensor": None, "frame_paths": [], "video_name": video_name}

    tensor = load_preprocessed_frames(video_name, output_dir)

    if tensor is None:
        return {"tensor": None, "frame_paths": [], "video_name": video_name}

    video_dir = Path(output_dir) / video_name
    jpg_files = sorted(video_dir.glob("*.jpg"))

    return {
        "tensor": tensor,
        "frame_paths": [str(p) for p in jpg_files],
        "video_name": video_name,
        "n_frames": len(tensor)
    }



ModuleNotFoundError: No module named 'face_extraction'

## 4. Predição

In [ ]:
def predict_and_explain_video(model, video_path, target_layer_name=None, 
                               n_explain_frames=6, cfg=None):
    cfg = cfg or CFG
    
    result = predict_video(model, video_path, cfg=cfg)
    
    if result.get("error"):
        print(f"[ERRO] {result['error']}")
        return result
    
    print(f"\\n[RESULTADO] Score: {result['video_score']:.4f} | Label: {result['video_label'].upper()}")
    
    plot_video_prediction(result)
    
    probs = result["frame_probs"]
    n_frames = len(probs)
    
    if n_frames == 0:
        print("[AVISO] Nenhum frame para explicar.")
        return result
    
    if result["video_label"] == "fake":
        top_indices = np.argsort(probs)[-n_explain_frames:][::-1]
    else:
        top_indices = np.argsort(probs)[:n_explain_frames]
    
    print(f"[EXPLICAÇÃO] Analisando {len(top_indices)} frames: {list(top_indices)}")
    
    frame_paths = result["frame_paths"]
    original_frames = load_original_frames([frame_paths[i] for i in top_indices])
    
    explanations = []
    for idx in top_indices:
        frame_tensor = result["tensor"][idx:idx+1]
        
        heatmap, pred_idx, prob = gc.make_gradcampp_vit_heatmap(
            model, frame_tensor, target_layer_name, pred_index=None
        )
        
        explanations.append({
            "frame_idx": int(idx),
            "frame_prob": float(probs[idx]),
            "heatmap": heatmap,
            "pred_label": "fake" if pred_idx == 1 else "real",
            "pred_prob": prob
        })
    
    plot_gradcam_explanations(original_frames, explanations, result["video_label"])
    
    return {**result, "explanations": explanations}



In [ ]:
def predict_video(model, video_path, output_dir=None, cfg=None):
    cfg = cfg or CFG

    pipeline = preprocess_video_for_prediction(video_path, output_dir)

    if pipeline["tensor"] is None:
        return {
            "video_path": str(video_path),
            "video_name": pipeline["video_name"],
            "error": "Nenhuma face detectada",
            "frame_probs": np.array([]),
            "video_score": None,
            "video_label": None,
            "n_frames": 0
        }

    tensor = pipeline["tensor"]

    frame_probs = model.predict(tensor, batch_size=cfg["batch_size"], verbose=0).flatten()

    video_score = aggregate_video(
        frame_probs,
        method=cfg["aggregation"],
        threshold=cfg["threshold"],
        trim_percent=cfg["trim_percent"]
    )

    video_label = classify_video(video_score, cfg["threshold"])

    return {
        "video_path": str(video_path),
        "video_name": pipeline["video_name"],
        "frame_paths": pipeline["frame_paths"],
        "frame_probs": frame_probs,
        "video_score": video_score,
        "video_label": video_label,
        "n_frames": len(frame_probs)
    }


def predict_videos_batch(model, video_paths, output_base_dir=None, cfg=None):
    cfg = cfg or CFG
    results = []

    for i, vp in enumerate(video_paths):
        print(f"\n[{i+1}/{len(video_paths)}] {Path(vp).name}")

        out_dir = None
        if output_base_dir:
            out_dir = Path(output_base_dir) / Path(vp).stem

        result = predict_video(model, vp, out_dir, cfg)
        results.append(result)

        if result.get("error"):
            print(f"   ⚠️  {result['error']}")
        else:
            print(f"   Score: {result['video_score']:.4f} | Label: {result['video_label'].upper()} | Frames: {result['n_frames']}")

    return results

In [ ]:
# @title Avaliação do modelo

def evaluate_predictions(results, true_labels_dict):
    y_true = []
    y_pred = []
    y_scores = []
    errors = []

    for res in results:
        name = res["video_name"]

        if res.get("error"):
            errors.append({"video": name, "error": res["error"]})
            continue

        if name not in true_labels_dict:
            errors.append({"video": name, "error": "Sem ground truth"})
            continue

        y_true.append(true_labels_dict[name])
        y_pred.append(1 if res["video_label"] == "fake" else 0)
        y_scores.append(res["video_score"])

    metrics = {
        "n_evaluated": len(y_true),
        "n_errors": len(errors),
        "accuracy": accuracy_score(y_true, y_pred) if y_true else None,
        "f1": f1_score(y_true, y_pred) if y_true else None,
    }

    if len(set(y_true)) > 1:
        metrics["auc"] = roc_auc_score(y_true, y_scores)

    return {"metrics": metrics, "errors": errors, "details": list(zip(y_true, y_pred, y_scores))}



In [ ]:
# @title Visualização de resultados

def plot_video_prediction(result, cfg=None, figsize=(14, 4)):
    cfg = cfg or CFG

    if result.get("error"):
        print(f"Erro: {result['error']}")
        return

    probs = result["frame_probs"]
    n = len(probs)

    fig, axes = plt.subplots(1, 3, figsize=figsize)

    colors = ["#dc2626" if p > cfg["threshold"] else "#16a34a" for p in probs]
    axes[0].bar(range(n), probs, color=colors, alpha=0.85, edgecolor="white", linewidth=0.5)
    axes[0].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2, label=f"threshold={cfg['threshold']}")
    axes[0].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5, label=f"{cfg['aggregation']}={result['video_score']:.3f}")
    axes[0].set_xlabel("Frame")
    axes[0].set_ylabel("Probabilidade FAKE")
    axes[0].set_title(f"Predições por Frame | {result['video_label'].upper()}")
    axes[0].legend(fontsize=9)
    axes[0].set_ylim(0, 1)

    axes[1].hist(probs, bins=min(20, n), color="#6366f1", alpha=0.7, edgecolor="white")
    axes[1].axvline(x=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[1].axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[1].set_xlabel("Probabilidade FAKE")
    axes[1].set_ylabel("Frequência")
    axes[1].set_title("Distribuição")

    axes[2].plot(range(n), probs, color="#8b5cf6", linewidth=1.5, alpha=0.8)
    axes[2].axhline(y=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    axes[2].axhline(y=result["video_score"], color="#3b82f6", linestyle="-", linewidth=2.5)
    axes[2].fill_between(range(n), 0, probs, where=(probs > cfg["threshold"]), alpha=0.2, color="#dc2626")
    axes[2].fill_between(range(n), 0, probs, where=(probs <= cfg["threshold"]), alpha=0.2, color="#16a34a")
    axes[2].set_xlabel("Frame (tempo)")
    axes[2].set_ylabel("Probabilidade FAKE")
    axes[2].set_title("Evolução Temporal")
    axes[2].set_ylim(0, 1)

    fig.suptitle(f"{result['video_name']} | Frames: {n}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def plot_aggregation_comparison(probs, threshold=0.5, trim_percent=10):
    methods = ["mean", "median", "vote", "trimmed_mean"]
    scores = [aggregate_video(probs, m, threshold, trim_percent) for m in methods]
    labels = [classify_video(s, threshold) for s in scores]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(methods, scores, color=colors, alpha=0.85, edgecolor="white", linewidth=2)
    ax.axhline(y=threshold, color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score Agregado")
    ax.set_title("Comparação de Métodos de Agregação")

    for bar, score, label in zip(bars, scores, labels):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.025,
                f"{score:.3f}\n({label.upper()})", ha="center", va="bottom",
                fontsize=11, fontweight="bold")

    plt.tight_layout()
    plt.show()


def plot_dataset_results(results, cfg=None, figsize=(12, 6)):
    cfg = cfg or CFG
    valid = [r for r in results if not r.get("error")]
    if not valid:
        print("Nenhum resultado válido.")
        return

    names = [r["video_name"][:22] for r in valid]
    scores = [r["video_score"] for r in valid]
    labels = [r["video_label"] for r in valid]
    colors = ["#dc2626" if l == "fake" else "#16a34a" for l in labels]

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.barh(range(len(names)), scores, color=colors, alpha=0.85, edgecolor="white")
    ax.axvline(x=cfg["threshold"], color="#f59e0b", linestyle="--", linewidth=2)
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=9)
    ax.set_xlabel("Score Agregado")
    ax.set_title(f"Resultados por Vídeo | Agregação: {cfg['aggregation']}")
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    for bar, score in zip(bars, scores):
        ax.text(score + 0.02, bar.get_y() + bar.get_height()/2, f"{score:.3f}",
                va="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    plt.show()

In [ ]:
# @title GRAD_CAM++ EXPLICAÇÕES

def plot_gradcam_explanations(original_frames, explanations, video_label, 
                               figsize_per_frame=(4, 4), alpha=0.5):
    n = len(explanations)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols * 2, figsize=(cols * 2 * figsize_per_frame[0], 
                                                       rows * figsize_per_frame[1]))
    if rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, (frame, exp) in enumerate(zip(original_frames, explanations)):
        row = i // cols
        col = (i % cols) * 2
        
        axes[row, col].imshow(frame)
        axes[row, col].set_title(
            f"Frame {exp['frame_idx']}\\nProb: {exp['frame_prob']:.3f}", 
            fontsize=10
        )
        axes[row, col].axis("off")
        
        heatmap = exp["heatmap"]
        overlay = gc.overlay_heatmap(frame, heatmap, alpha=alpha)
        
        axes[row, col + 1].imshow(overlay)
        axes[row, col + 1].set_title(
            f"Grad-CAM++\\n{exp['pred_label'].upper()} ({exp['pred_prob']:.3f})",
            fontsize=10,
            color="#dc2626" if exp["pred_label"] == "fake" else "#16a34a"
        )
        axes[row, col + 1].axis("off")
    
    for i in range(n, rows * cols):
        row = i // cols
        col = (i % cols) * 2
        if row < axes.shape[0] and col < axes.shape[1]:
            axes[row, col].axis("off")
        if row < axes.shape[0] and col + 1 < axes.shape[1]:
            axes[row, col + 1].axis("off")
    
    fig.suptitle(
        f"Explicações Grad-CAM++ | Vídeo: {video_label.upper()}",
        fontsize=14, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.show()



def plot_batch_explanations_summary(results, figsize=(14, 6)):
    valid = [r for r in results if not r.get("error") and "explanations" in r]
    if not valid:
        print("Nenhum resultado com explicações.")
        return
    
    n = len(valid)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 3.5))
    if rows == 1:
        axes = axes.reshape(1, -1) if n > 1 else [axes]
    
    for i, res in enumerate(valid):
        row = i // cols
        col = i % cols
        
        heatmaps = [e["heatmap"] for e in res["explanations"]]
        avg_heatmap = np.mean(heatmaps, axis=0)
        
        axes[row, col].imshow(avg_heatmap, cmap="jet")
        axes[row, col].set_title(
            f"{res['video_name'][:20]}\\n"
            f"{res['video_label'].upper()} ({res['video_score']:.3f})",
            fontsize=10
        )
        axes[row, col].axis("off")
    
    for i in range(n, rows * cols):
        row = i // cols
        col = i % cols
        if row < axes.shape[0] and col < axes.shape[1]:
            axes[row, col].axis("off")
    
    fig.suptitle("Heatmap Médio por Vídeo (Grad-CAM++)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


## 5. Execução da pipeline

In [ ]:
print(f"Modelo: {CFG['model_path']}")
print(f"Dataset: {CFG['dataset']}")
print(f"Backbone: {'Fine-tuned' if CFG['use_finetuned'] else 'Original'}")
print(f"Threshold: {CFG['threshold']}")
print(f"Agregação: {CFG['aggregation']} (trim={CFG['trim_percent']}%)")

model = keras.models.load_model(CFG['model_path'])
print(f"\n✓ Modelo: {model.input_shape} → {model.output_shape}")

backbone_path = ("siglip2_base_patch16_224_finetune_backbone.keras"
if CFG['use_finetuned']
else "siglip2_base_patch16_224_original_backbone.keras"
)

if os.path.exists(backbone_path):
    backbone = keras.models.load_model(backbone_path)
    print(f"✓ Backbone: {backbone.input_shape} → {backbone.output_shape}")
else:
    print(f"⚠️  Backbone não encontrado, baixando...")
    backbone = keras_hub.models.SigLIPBackbone.from_preset("siglip2_base_patch16_224")
    backbone.trainable = False
    backbone.save(backbone_path)

os.makedirs(CFG['pred_output_dir'], exist_ok=True)
print(f"✓ Saída: {CFG['pred_output_dir']}")

# result = predict_and_explain_video(
#     model,
#     "/caminho/video_teste.mp4",
#     target_layer_name=None,
#     n_explain_frames=6
# )

# plot_aggregation_comparison(result["frame_probs"])

Input shape: (None, 768)
Model output shape: (None, 1)
Número de camadas: 10
0: dense_4 (Dense)
1: batch_normalization_3 (BatchNormalization)
2: dropout_228 (Dropout)
3: dense_5 (Dense)
4: batch_normalization_4 (BatchNormalization)
5: dropout_229 (Dropout)
6: dense_6 (Dense)
7: batch_normalization_5 (BatchNormalization)
8: dropout_230 (Dropout)
9: dense_7 (Dense)


In [ ]:
# model = keras.models.load_model(CFG["model_path"])
# 
# # 8.1 — Um vídeo com explicação
# result = predict_and_explain_video(
#     model, 
#     "/caminho/video_teste.mp4",
#     target_layer_name="transformer_block_11",  # ou None para auto-detect
#     n_explain_frames=6
# )
# 
# # 8.2 — Comparar agregação (mantido igual)
# plot_aggregation_comparison(result["frame_probs"])
# 
# # 8.3 — Lote de vídeos com explicação
# video_list = glob("/caminho/testes/*.mp4")
# results = predict_and_explain_batch(
#     model, 
#     video_list,
#     target_layer_name="transformer_block_11",
#     n_explain_frames=4
# )
# 
# Resumo comparativo
# plot_batch_explanations_summary(results)


